# Configuration

All settings of the analysis in one object, `cfg`. The analysis notebooks read everything from here (for
example `cfg.K`, the number of clusters), so a setting changed here changes it everywhere.

This notebook is run by `setup.ipynb` and does not need to be run on its own.

## Paths

The project folder is found by looking for `requirements.txt`, so the paths work on any computer.

In [ ]:
from pathlib import Path
from types import SimpleNamespace

cfg = SimpleNamespace()

cfg.PROJECT_ROOT = Path.cwd()
while not (cfg.PROJECT_ROOT / "requirements.txt").exists():
    cfg.PROJECT_ROOT = cfg.PROJECT_ROOT.parent

cfg.DATA_DIR = cfg.PROJECT_ROOT / "notebooks" / "01_data_pipeline"
cfg.CLINICAL_FILE = cfg.DATA_DIR / "oasis_cross-sectional-5708aa0a98d82080.xlsx"
cfg.RESULTS_DIR = cfg.PROJECT_ROOT / "results"

## Data

- Five MRI image types, each reduced to 110 principal components of its HOG features.
- Clinical variables used for clustering:
  - quantitative: Age, MMSE, eTIV and nWBV. ASF is left out because it is computed from the intracranial volume
    and correlates −0.99 with eTIV, so it would count head size twice and make the covariance matrix almost
    singular.
  - multiclass: Education, SES and sex. Education and SES are coded in a few categories and sex in two, so they
    are compared by whether two patients share the same category (Hamming distance).
- Variables used to interpret and compare the clusters (the same for every configuration): Age, MMSE, nWBV, eTIV
  and CDR, compared with |Cliff's δ|, and Education, SES and sex, compared with the Total Variation Distance. They
  are treated as categories here too, as in the clustering.

In [ ]:
cfg.IMAGE_TYPES = ["cor", "gfc_sag", "gfc_tra", "masked_tra", "sbj_sag"]
cfg.IMAGE_NAMES = {
    "cor": "coronal",
    "gfc_sag": "atlas sagittal",
    "gfc_tra": "atlas transverse",
    "masked_tra": "masked transverse",
    "sbj_sag": "native sagittal",
}
cfg.N_PCS = 110
cfg.EXPECTED_N = 216

cfg.SEX = "sex"  # 1 = male, 0 = female
cfg.CDR = "CDR"
cfg.CLINICAL_QUANT = ["Age", "MMSE", "eTIV", "nWBV"]
cfg.CLINICAL_MULTICLASS = ["Educ", "SES", cfg.SEX]

cfg.PSI_ORDINAL = ["Age", "MMSE", "nWBV", "eTIV", "CDR"]
cfg.PSI_NOMINAL = ["Educ", "SES", cfg.SEX]

## Clustering settings

| Setting | Value | Note |
|---|---|---|
| `K_CANDIDATES`, `K_SELECTION_CONFIGS` | k = 2 to 5, on C1, M1 and I1 | used to choose k |
| `K` | 4 | chosen in `analysis/01_choose_k` |
| `RANDOM_STATE` | 123 | |
| `FRAC_SAMPLE_SIZE` | 1.0 | no subsampling: PAM on the full distance matrix |
| `ROBUST_METHOD`, `ALPHA` | trimmed, 0.05 | robust covariance estimate |
| `PAM` | pam, build, 100 iterations | |
| `KMEANS_N_INIT` | 10 | several k-means starts, so C1-b does not depend on one seed |
| `N_STABILITY_REFITS` | 30 | refits in the stability check |

In [ ]:
cfg.K_CANDIDATES = [2, 3, 4, 5]
cfg.K_SELECTION_CONFIGS = ["C1", "M1", "I1"]
cfg.K = 4

cfg.RANDOM_STATE = 123
cfg.FRAC_SAMPLE_SIZE = 1.0
cfg.ROBUST_METHOD = "trimmed"
cfg.ALPHA = 0.05
cfg.PAM = {"method": "pam", "init": "build", "max_iter": 100}
cfg.KMEANS_N_INIT = 10

cfg.N_STABILITY_REFITS = 30

## The nine configurations

Each quantitative block is written as `(label, columns, distance)`; `multiclass` lists the variables of the
multiclass block, which always uses the Hamming distance.

- Images: one block per image type, Euclidean distance (the PCA scores are already on a common scale).
- Clinical quantitative variables (Age, MMSE, eTIV, nWBV): one block with robust Mahalanobis distance, which does
  not depend on the units of the variables (otherwise eTIV, in cm³, dominates). It is never in the same block as
  the images.
- Education, SES and sex: the multiclass block (Hamming), in every configuration that uses clinical data.
- C1-b uses k-means, which has no blocks: Age, MMSE, eTIV and nWBV are standardised and Education, SES and sex are
  one-hot encoded (one 0/1 column per category), so it uses the same information as C1.
- I2-b uses k-means on the 550 image PCs as they are (they are already on a common scale), so it uses the same
  information as I1 and I2.
- M3 uses the distance (RelMS or G-Gower) of whichever of M1 and M2 has the higher PSI; this is decided in
  `analysis/02_compare_configurations`.

| ID | Feature space | Method | What it tests |
|---|---|---|---|
| C1 | Clinical | Robust G-Gower + PAM | clinical baseline |
| C1-b | Clinical | k-means | effect of the algorithm on the same features |
| C2 | Clinical + CDR | Robust G-Gower + PAM | severity as part of the profile |
| I1 | Images | RelMS + PAM | image-only structure |
| I2 | Images | G-Gower + PAM | RelMS vs. G-Gower on images |
| I2-b | Images | k-means | effect of the algorithm on images |
| M1 | Images + clinical | RelMS + PAM | added value of images |
| M2 | Images + clinical | Robust G-Gower + PAM | same, with G-Gower |
| M3 | Images + clinical + CDR | best of M1/M2 | full profile including severity |

In [ ]:
IMAGE_BLOCKS = [(t, f"image:{t}", "euclidean") for t in cfg.IMAGE_TYPES]
CLINICAL_BLOCK = ("clinical", cfg.CLINICAL_QUANT, "robust_mahalanobis")
CLINICAL_CDR_BLOCK = ("clinical+CDR", cfg.CLINICAL_QUANT + [cfg.CDR], "robust_mahalanobis")

cfg.CONFIGS = {
    "C1": {
        "description": "Clinical baseline",
        "feature_space": "Clinical",
        "algorithm": "kmedoids",
        "metric": "ggower",
        "quant_blocks": [CLINICAL_BLOCK],
        "binary": [],
        "multiclass": cfg.CLINICAL_MULTICLASS,
    },
    "C1-b": {
        "description": "Effect of the algorithm on the same features",
        "feature_space": "Clinical",
        "algorithm": "kmeans",
        "kmeans_standardise": cfg.CLINICAL_QUANT,
        "kmeans_onehot": cfg.CLINICAL_MULTICLASS,
        "kmeans_raw": [],
    },
    "C2": {
        "description": "Severity as part of the profile",
        "feature_space": "Clinical + CDR",
        "algorithm": "kmedoids",
        "metric": "ggower",
        "quant_blocks": [CLINICAL_CDR_BLOCK],
        "binary": [],
        "multiclass": cfg.CLINICAL_MULTICLASS,
    },
    "I1": {
        "description": "Image-only structure",
        "feature_space": "Images",
        "algorithm": "kmedoids",
        "metric": "relms",
        "quant_blocks": IMAGE_BLOCKS,
        "binary": [],
        "multiclass": [],
    },
    "I2": {
        "description": "RelMS vs. G-Gower on images",
        "feature_space": "Images",
        "algorithm": "kmedoids",
        "metric": "ggower",
        "quant_blocks": IMAGE_BLOCKS,
        "binary": [],
        "multiclass": [],
    },
    "I2-b": {
        "description": "Effect of the algorithm on images",
        "feature_space": "Images",
        "algorithm": "kmeans",
        "kmeans_standardise": [],
        "kmeans_onehot": [],
        "kmeans_raw": [f"image:{t}" for t in cfg.IMAGE_TYPES],
    },
    "M1": {
        "description": "Added value of images",
        "feature_space": "Images + clinical",
        "algorithm": "kmedoids",
        "metric": "relms",
        "quant_blocks": IMAGE_BLOCKS + [CLINICAL_BLOCK],
        "binary": [],
        "multiclass": cfg.CLINICAL_MULTICLASS,
    },
    "M2": {
        "description": "Same, with G-Gower",
        "feature_space": "Images + clinical",
        "algorithm": "kmedoids",
        "metric": "ggower",
        "quant_blocks": IMAGE_BLOCKS + [CLINICAL_BLOCK],
        "binary": [],
        "multiclass": cfg.CLINICAL_MULTICLASS,
    },
    "M3": {
        "description": "Full profile including severity",
        "feature_space": "Images + clinical + CDR",
        "algorithm": "kmedoids",
        "metric": None,  # the better of M1 / M2 under the PSI, set in analysis/02
        "quant_blocks": IMAGE_BLOCKS + [CLINICAL_CDR_BLOCK],
        "binary": [],
        "multiclass": cfg.CLINICAL_MULTICLASS,
    },
}